# 02. Inspect feature distributions and coverage
The full model retains 52 features. The original six are retained, with explicit
level, variability, regression trend, long-window level and persistence summaries.
One-hour and six-hour optical windows are modelling choices, not standards.
FEC adds error-interval frequency. Temperature retains level, slope and variability.
All references use training only. No importance threshold removes any feature.
See METHOD.md for each feature definition, source and assumption.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation
from optical_anomaly.workflow import run_split

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
split = run_split(settings)
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    split.train_end,
    split.calibration_end,
    split.validation_end,
    split.test_end,
]
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Run:", RUN.resolve())

In [ ]:
from optical_anomaly.workflow import prepare_canonical
from optical_anomaly.features import FeatureEngineer
from optical_anomaly.multivariate import (
    MultivariateFeatures,
    FEATURE_SETS,
    columns_for,
)

canonical_path = prepare_canonical(RUN)
entity = pd.read_parquet(RUN / "topology.parquet").entity_id.iloc[0]
telemetry = pd.read_parquet(
    canonical_path,
    filters=[("entity_id", "==", entity), ("timestamp", "<", split.train_end)],
)
engineer = MultivariateFeatures(
    FeatureEngineer(
        interval_minutes=settings["generator"]["interval_minutes"],
        **settings["features"],
    )
).fit(telemetry)
features = engineer.transform(telemetry)
for feature_set in FEATURE_SETS:
    columns = columns_for(feature_set)
    print(
        feature_set,
        len(columns),
        "features; complete coverage:",
        features[columns].notna().all(axis=1).mean(),
    )
    display(features[columns].describe())
features[columns_for("temperature")].hist(bins=30, figsize=(16, 16))
plt.tight_layout()
plt.show()

A missing reading restarts the rolling history. No imputation borrows future
values. Normalisation requires a representative local baseline; portability is not
zero-shot transfer. Unknown entities abstain. Seasonal correction is a modelling
assumption: compare `seasonal: false` in a new run before deciding it helps.
Entropy of constant/saturated bins can decrease during a fault; Isolation Forest
can use either direction. CUSUM uses the fixed training-normalized reference zero so its target does not
follow sustained loss. Daily references require at least two calendar days.

## Observation quality and available history
These diagnostics explain abstention; they are not optical model inputs.
Long means use the available contiguous history, from the short-window length
up to the configured maximum. A complete six-hour window is not required.
The `temperature` key identifies the full model including temperature.

In [ ]:
from optical_anomaly.diagnostics import observation_quality

quality = observation_quality(
    telemetry, settings["generator"]["interval_minutes"],
    settings["features"]["window"],
)
quality.to_csv(REPORT / "example_observation_quality.csv", index=False)
display(quality.describe(include="all"))
display(quality.loc[~quality.current_observed].head(12))
availability = []
for name in FEATURE_SETS:
    ready = features[columns_for(name)].notna().all(axis=1)
    availability.append({
        "feature_set": name,
        "ready_fraction": ready.mean(),
        "first_ready_time": features.loc[ready, "timestamp"].min(),
    })
display(pd.DataFrame(availability))
